In [ ]:
import os, sys, shutil, zipfile, time
from pathlib import Path
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")


In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

os.chdir("/kaggle/working")

# 1. Extract any zip files found in /kaggle/input
for z in Path("/kaggle/input").rglob("*.zip"):
    print(f"Extracting {z} into /kaggle/working...")
    with zipfile.ZipFile(z, "r") as zf:
        zf.extractall("/kaggle/working")

# 2. Find and copy directories from capstone-code
code_dirs = [p for p in Path("/kaggle/input").rglob("capstone-code") if p.is_dir()]
if code_dirs:
    code_in = code_dirs[0]
    for item in ["src", "configs", "tests", "results"]:
        src_item = code_in / item
        dst_item = Path("/kaggle/working") / item
        if src_item.is_dir() and not dst_item.exists():
            shutil.copytree(src_item, dst_item)
            print(f"Copied {item} from {src_item} to {dst_item}")

# 3. Symlink dataset dev partitions and processed splits
p_dev = None
for p in Path("/kaggle").rglob("client_00.parquet"):
    if "partitions" in str(p):
        p_dev = p.parent.parent if p.parent.name.startswith("seed_") else p.parent
        break

pr_dev = None
for p in Path("/kaggle").rglob("test.parquet"):
    if "processed" in str(p):
        pr_dev = p.parent
        break

if p_dev and not Path("/kaggle/working/data/partitions/dev").exists():
    os.makedirs("/kaggle/working/data/partitions", exist_ok=True)
    os.system("ln -sfn " + str(p_dev) + " /kaggle/working/data/partitions/dev")
    print(f"Linked partitions/dev -> {p_dev}")

if pr_dev and not Path("/kaggle/working/data/processed/dev").exists():
    os.makedirs("/kaggle/working/data/processed", exist_ok=True)
    os.system("ln -sfn " + str(pr_dev) + " /kaggle/working/data/processed/dev")
    print(f"Linked processed/dev -> {pr_dev}")

for m in Path("/kaggle").rglob("feature_stats_dev.json"):
    os.makedirs("/kaggle/working/data/metadata", exist_ok=True)
    shutil.copy2(m, "/kaggle/working/data/metadata/feature_stats_dev.json")
    print(f"Linked metadata from {m}")
    break

print("Working dir contents:", os.listdir("/kaggle/working"))
assert Path("/kaggle/working/src").exists(), "src directory not found!"
print("Setup complete!")


In [ ]:
import os
print("Running test suite...")
ret = os.system("PYTHONPATH=. pytest tests/test_determinism.py tests/test_harness.py tests/test_metrics.py tests/test_report_hygiene.py tests/test_variants.py")
assert ret == 0, f"Test suite failed with exit code {ret}"
print("All tests passed successfully!")


In [ ]:
import os
print("Running Phase E4.2a Benchmark...")
ret = os.system("PYTHONPATH=. python src/experiments/run_phase_e4_2a.py --workers 8 --rounds 30 --output-dir results/runs/phase_e4_2a")
assert ret == 0, f"Benchmark failed with exit code {ret}"
print("Benchmark finished successfully!")


In [ ]:
import os
print("Packaging results...")
os.system("zip -r /kaggle/working/phase_e4_2a_results.zip results/runs/phase_e4_2a/")
print("Results ready at /kaggle/working/phase_e4_2a_results.zip")
